In [1]:
#!/usr/bin/env python3
"""
DGH-XH: Beijing Multi-Site AQI (PM2.5)
==========================================
Dataset : UCI Beijing Multi-Site Air Quality Data
          Kaggle: sid321axn/beijing-multisite-airquality-data-set
Target  : PM2.5 (μg/m³)
Stations: 12 (KNN k=4 graph)
Period  : 2013-03-01 to 2017-02-28
Split   : test from 2016-09-01 onwards
Wind    : Meteorological direction (wd, degrees) + speed (WSPM)
          Converted to u/v via: u = -sin(θ)*spd, v = -cos(θ)*spd
"""

# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import os, warnings, joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)
import xgboost as xgb
from xgboost import XGBRegressor
from IPython.display import display

warnings.filterwarnings("ignore")

# ============================================================
# CELL 2 — CONFIG  (only this block differs from Chennai)
# ============================================================
# Kaggle input path for: sid321axn/beijing-multisite-airquality-data-set
BEIJING_DATA_DIR = "/kaggle/input/datasets/sid321axn/beijing-multisite-airquality-data-set"
OUT_DIR          = "dghxh_beijing_aqi"
SPLIT_TIME       = pd.Timestamp("2016-09-01 00:00:00")
L                = 24          # lookback hours
H_LIST           = [1, 3, 6, 12, 24]
TAU_LIST         = [0, 1, 2, 3, 4, 6]
VALID_FRAC       = 0.20
GRAPH_K          = 4
RUN_TUNED        = True
JSO_POP          = 6   # 48 evals (was 180): saves ~1.5h/horizon
JSO_ITERS        = 8   # reduced from 15
RANDOM_SEED      = 1

# ── Beijing station coordinates (from UCI metadata) ──────────
STATION_COORDS = {
    "Aotizhongxin":  (40.00, 116.41),
    "Changping":     (40.22, 116.23),
    "Dingling":      (40.29, 116.22),
    "Dongsi":        (39.93, 116.42),
    "Guanyuan":      (39.93, 116.34),
    "Gucheng":       (39.91, 116.18),
    "Huairou":       (40.38, 116.63),
    "Nongzhanguan":  (39.94, 116.46),
    "Shunyi":        (40.13, 116.65),
    "Tiantan":       (39.88, 116.41),
    "Wanliu":        (39.99, 116.30),
    "Wanshouxigong": (39.88, 116.36),
}

# ============================================================
# CELL 3 — DATASET LOADER  (Beijing-specific)
# ============================================================
def met_wind_to_uv(wd_deg_series, wspd_series):
    """
    Convert meteorological wind direction + speed to u/v components.
    Meteorological convention: wd = direction wind COMES FROM, clockwise from N.
    ERA5 convention (what our gate uses): u = eastward, v = northward component
    of the direction wind is GOING TO.

    Formula:
        u = -sin(θ) * spd   (east component of wind motion)
        v = -cos(θ) * spd   (north component of wind motion)
    where θ is the direction the wind is coming FROM in radians.

    Physical justification: cos(θ_wind_going - β_edge) in the gate computes
    alignment between wind motion direction and the edge bearing, which is exactly
    what we want for advection — the same as using ERA5 u/v with arctan2(v, u).
    """
    wd_deg = pd.to_numeric(wd_deg_series, errors="coerce").fillna(0.0)
    wspd   = pd.to_numeric(wspd_series, errors="coerce").fillna(0.0)
    theta  = np.radians(wd_deg.to_numpy())
    u = (-np.sin(theta) * wspd.to_numpy()).astype(np.float32)
    v = (-np.cos(theta) * wspd.to_numpy()).astype(np.float32)
    return u, v

def load_beijing_aqi():
    """
    Load all 12 Beijing station CSV files, merge, compute u/v wind,
    and return a unified long-format DataFrame matching the Chennai schema.
    Target: PM2.5
    Features returned: pm25, temp, pres, dewp, u_wind, v_wind
    """
    dfs = []
    # Kaggle sometimes nests files one level deeper — find where CSVs actually live
    csv_dir = BEIJING_DATA_DIR
    for candidate in [
        BEIJING_DATA_DIR,
        os.path.join(BEIJING_DATA_DIR, "PRSA_Data_20130301-20170228"),
        os.path.join(BEIJING_DATA_DIR, "archive"),
    ]:
        if os.path.isdir(candidate) and any(f.endswith(".csv") for f in os.listdir(candidate)):
            csv_dir = candidate
            break
    print(f"  Reading CSVs from: {csv_dir}")
    # File naming pattern in the UCI Kaggle dataset:
    # PRSA_Data_Aotizhongxin_20130301_20170228.csv  etc.
    for fname in sorted(os.listdir(csv_dir)):
        if not fname.endswith(".csv"):
            continue
        # Extract station name from filename
        parts = fname.replace(".csv", "").split("_")
        # Pattern: PRSA_Data_<StationName>_<start>_<end>
        station = parts[2] if len(parts) >= 3 else fname.replace(".csv", "")

        if station not in STATION_COORDS:
            print(f"  Skipping unrecognised station file: {fname}")
            continue

        df = pd.read_csv(os.path.join(csv_dir, fname))

        # ── Build timestamp ─────────────────────────────────
        df["timestamp"] = pd.to_datetime(
            df[["year", "month", "day", "hour"]].rename(
                columns={"year": "year", "month": "month",
                         "day": "day", "hour": "hour"}
            )
        )

        # ── Wind direction: may be string "NE" etc. or numeric ──
        # In the standard UCI dataset, 'wd' is a string direction
        # Convert to degrees first
        wd_str_to_deg = {
            "N": 0.0, "NNE": 22.5, "NE": 45.0, "ENE": 67.5,
            "E": 90.0, "ESE": 112.5, "SE": 135.0, "SSE": 157.5,
            "S": 180.0, "SSW": 202.5, "SW": 225.0, "WSW": 247.5,
            "W": 270.0, "WNW": 292.5, "NW": 315.0, "NNW": 337.5,
        }
        if df["wd"].dtype == object:
            df["wd_deg"] = df["wd"].map(wd_str_to_deg).fillna(0.0)
        else:
            df["wd_deg"] = pd.to_numeric(df["wd"], errors="coerce").fillna(0.0)

        u, v = met_wind_to_uv(df["wd_deg"], df["WSPM"])
        df["u_wind"] = u
        df["v_wind"] = v

        # ── Rename to standard schema ────────────────────────
        df["station_id"]   = station
        df["lat"]          = STATION_COORDS[station][0]
        df["lon"]          = STATION_COORDS[station][1]
        df["pm25"]         = pd.to_numeric(df["PM2.5"], errors="coerce")
        df["temp"]         = pd.to_numeric(df["TEMP"],  errors="coerce")
        df["pres"]         = pd.to_numeric(df["PRES"],  errors="coerce")
        df["dewp"]         = pd.to_numeric(df["DEWP"],  errors="coerce")

        dfs.append(df[[
            "timestamp", "station_id", "lat", "lon",
            "pm25", "temp", "pres", "dewp", "u_wind", "v_wind"
        ]])

    df_all = pd.concat(dfs, ignore_index=True).sort_values(
        ["timestamp", "station_id"]
    ).reset_index(drop=True)

    print(f"Loaded {len(df_all):,} rows, "
          f"{df_all.station_id.nunique()} stations, "
          f"period {df_all.timestamp.min()} → {df_all.timestamp.max()}")
    return df_all

# ============================================================
# CELL 4 — ALL SHARED HELPER FUNCTIONS (identical to Chennai)
# ============================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2.0*R*np.arcsin(np.sqrt(a))

def bearing_radians(lat1, lon1, lat2, lon2):
    lat1,lon1,lat2,lon2 = map(np.radians,[lat1,lon1,lat2,lon2])
    dlon = lon2-lon1
    y = np.sin(dlon)*np.cos(lat2)
    x = np.cos(lat1)*np.sin(lat2)-np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    return np.arctan2(y, x)

def flatten_window_per_node(X):
    S,Lx,N,F = X.shape
    return X.transpose(0,2,1,3).reshape(S*N, Lx*F)

def compute_tail_metrics(y_true, y_pred, percentiles=(90,95,99)):
    rows = []
    for p in percentiles:
        thr = np.percentile(y_true, p)
        idx = y_true >= thr
        if idx.sum() == 0:
            rows.append((p, thr, np.nan, np.nan, 0))
        else:
            rows.append((p, thr,
                mean_absolute_error(y_true[idx], y_pred[idx]),
                np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])),
                int(idx.sum())))
    return rows

def compute_mfb_nmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mfb  = np.mean(2.0*(y_pred-y_true)/(y_pred+y_true+1e-8))
    nmse = np.sum((y_pred-y_true)**2)/(np.sum(y_pred*y_true)+1e-8)
    return mfb, nmse

def build_fill_values_from_train_timeline(X_train_timeline):
    fill_values = np.nanmedian(X_train_timeline, axis=0)
    global_fill = np.nanmedian(
        X_train_timeline.reshape(-1, X_train_timeline.shape[-1]), axis=0)
    global_fill = np.where(np.isnan(global_fill), 0.0, global_fill)
    for n in range(fill_values.shape[0]):
        for f in range(fill_values.shape[1]):
            if np.isnan(fill_values[n,f]):
                fill_values[n,f] = global_fill[f]
    return np.where(np.isnan(fill_values), 0.0, fill_values).astype(np.float32)

def impute_windows(X, fill_values):
    X_imp = X.copy().astype(np.float32)
    S,Lx,N,F = X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                series = pd.Series(X_imp[s,:,n,f], dtype="float32")
                series = series.ffill().bfill()
                arr = series.to_numpy(dtype=np.float32)
                if np.isnan(arr).any():
                    arr = np.where(np.isnan(arr), fill_values[n,f], arr)
                X_imp[s,:,n,f] = arr
    return X_imp

def build_nodes_from_coords(station_ids, coord_dict):
    """Build nodes DataFrame from station_id → (lat, lon) dict."""
    rows = [{"station_id": sid,
             "lat": coord_dict[sid][0],
             "lon": coord_dict[sid][1],
             "node_id": i}
            for i, sid in enumerate(station_ids)]
    return pd.DataFrame(rows)

def build_edges_from_nodes(nodes_df, k=4):
    coords = nodes_df[["lat","lon"]].to_numpy(dtype=float)
    N = len(coords)
    D = np.zeros((N,N))
    for i in range(N):
        D[i,:] = haversine_km_vec(
            coords[i,0], coords[i,1], coords[:,0], coords[:,1])
    sigma = np.median(D[D>0])
    edges = []
    for i in range(N):
        nn = np.argsort(D[i])[1:min(k+1,N)]
        for j in nn:
            w = np.exp(-(D[i,j]**2)/(2.0*sigma**2))
            edges.append((i, j, float(w), float(D[i,j])))
    return pd.DataFrame(edges, columns=["src","dst","w_dist","dist_km"])

def build_static_adj(nodes_df, k=4):
    coords = nodes_df[["lat","lon"]].to_numpy(dtype=float)
    N = len(coords)
    D = np.zeros((N,N))
    for i in range(N):
        D[i,:] = haversine_km_vec(
            coords[i,0], coords[i,1], coords[:,0], coords[:,1])
    sigma = np.median(D[D>0])
    A = np.zeros((N,N), dtype=np.float32)
    for i in range(N):
        nn = np.argsort(D[i])[1:min(k+1,N)]
        for j in nn:
            A[i,j] = np.exp(-(D[i,j]**2)/(2.0*sigma**2))
    rs = A.sum(axis=1, keepdims=True)
    return np.divide(A, rs, out=np.zeros_like(A), where=rs>0)

def build_dynamic_adj(u_t, v_t, ctx, alpha=4.0, eps=0.05):
    theta_w = np.arctan2(v_t[ctx["src"]], u_t[ctx["src"]])
    align   = np.cos(theta_w - ctx["edge_bearing"])
    gate    = eps + (1.0-eps)/(1.0+np.exp(-alpha*align))
    w_dyn   = ctx["w_dist"] * gate
    A = np.zeros((ctx["N"], ctx["N"]), dtype=np.float32)
    A[ctx["src"], ctx["dst"]] = w_dyn.astype(np.float32)
    rs = A.sum(axis=1, keepdims=True)
    return np.divide(A, rs, out=np.zeros_like(A), where=rs>0)

def make_graph_features_dynamic(X, ctx, tau=0, alpha=4.0, eps=0.05):
    S,Lx,N,F = X.shape
    Z = np.zeros((S,N,2*F), dtype=np.float32)
    for s in range(S):
        x_now = X[s,-1]
        x_lag = X[s,-1-tau] if tau > 0 else x_now
        A = build_dynamic_adj(x_now[:,ctx["u_idx"]], x_now[:,ctx["v_idx"]],
                              ctx, alpha=alpha, eps=eps)
        Z[s] = np.concatenate([x_now, A @ x_lag], axis=-1)
    return Z

def make_graph_features_static(X, A_static, tau=0):
    S,Lx,N,F = X.shape
    Z = np.zeros((S,N,2*F), dtype=np.float32)
    for s in range(S):
        x_now = X[s,-1]
        x_lag = X[s,-1-tau] if tau > 0 else x_now
        Z[s] = np.concatenate([x_now, A_static @ x_lag], axis=-1)
    return Z

def summarize_regression(y_true, y_pred):
    return {"MAE":  float(mean_absolute_error(y_true, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2":   float(r2_score(y_true, y_pred))}

def train_val_split_time_order(X, Y, M, times, frac=0.2):
    n = X.shape[0]
    val_size = max(1, int(np.ceil(frac*n)))
    idx_tr = np.arange(0, n-val_size)
    idx_va = np.arange(n-val_size, n)
    tr = {"X":X[idx_tr],"Y":Y[idx_tr],"M":M[idx_tr],"times":times[idx_tr]}
    va = {"X":X[idx_va],"Y":Y[idx_va],"M":M[idx_va],"times":times[idx_va]}
    return tr, va

def fit_flat_regressor(model, Xtr, Ytr, Mtr, Xte, Yte, Mte):
    Xtr2 = flatten_window_per_node(Xtr)
    Xte2 = flatten_window_per_node(Xte)
    ytr = Ytr.reshape(-1); yte = Yte.reshape(-1)
    mtr = (Mtr.reshape(-1)>0.5); mte = (Mte.reshape(-1)>0.5)
    model.fit(Xtr2[mtr], ytr[mtr])
    return {"y_true":yte[mte], "y_pred":model.predict(Xte2)[mte], "model":model}

def fit_static_graph_regressor(model, Xtr, Ytr, Mtr, Xte, Yte, Mte, A_static, tau):
    Ztr = make_graph_features_static(Xtr, A_static, tau=tau)
    Zte = make_graph_features_static(Xte, A_static, tau=tau)
    Xtr2 = Ztr.reshape(-1, Ztr.shape[-1]); Xte2 = Zte.reshape(-1, Zte.shape[-1])
    ytr = Ytr.reshape(-1); yte = Yte.reshape(-1)
    mtr = (Mtr.reshape(-1)>0.5); mte = (Mte.reshape(-1)>0.5)
    model.fit(Xtr2[mtr], ytr[mtr])
    return {"y_true":yte[mte], "y_pred":model.predict(Xte2)[mte], "model":model}

def fit_huber_graph(Xtr, Ytr, Mtr, Xte, Yte, Mte, ctx,
                    tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4):
    Ztr = make_graph_features_dynamic(Xtr, ctx, tau=tau, alpha=alpha, eps=eps)
    Zte = make_graph_features_dynamic(Xte, ctx, tau=tau, alpha=alpha, eps=eps)
    Xtr2 = Ztr.reshape(-1, Ztr.shape[-1]); Xte2 = Zte.reshape(-1, Zte.shape[-1])
    ytr = Ytr.reshape(-1); yte = Yte.reshape(-1)
    mtr = (Mtr.reshape(-1)>0.5); mte = (Mte.reshape(-1)>0.5)
    huber = Pipeline([("sc", StandardScaler()),
                      ("h", HuberRegressor(epsilon=1.35, alpha=huber_alpha, max_iter=500))])
    huber.fit(Xtr2[mtr], ytr[mtr])
    ptr_all = huber.predict(Xtr2)
    pte_all = huber.predict(Xte2)
    return {"y_true":yte[mte], "y_pred":pte_all[mte],
            "pred_train_all":ptr_all, "pred_test_all":pte_all,
            "y_train_all":ytr, "y_test_all":yte,
            "mask_train":mtr, "mask_test":mte, "model":huber}

def fit_huber_hybrid(Xtr, Ytr, Mtr, Xte, Yte, Mte, ctx,
                     tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4,
                     xgb_params=None, num_boost_round=400, seed=42,
                     w_mid=2.0, w_danger=5.0, w_tail=10.0):
    base = fit_huber_graph(Xtr, Ytr, Mtr, Xte, Yte, Mte, ctx,
                           tau=tau, alpha=alpha, eps=eps, huber_alpha=huber_alpha)
    ytr_all = base["y_train_all"]; mtr = base["mask_train"]; mte = base["mask_test"]
    res_train = ytr_all - base["pred_train_all"]
    Xg_tr = np.asarray(flatten_window_per_node(Xtr), dtype=np.float32)
    Xg_te = np.asarray(flatten_window_per_node(Xte), dtype=np.float32)
    t90 = np.percentile(ytr_all[mtr], 90)
    t95 = np.percentile(ytr_all[mtr], 95)
    t99 = np.percentile(ytr_all[mtr], 99)
    w = np.ones_like(ytr_all[mtr], dtype=np.float32)
    w[ytr_all[mtr] >= t90] = w_mid
    w[ytr_all[mtr] >= t95] = w_danger
    w[ytr_all[mtr] >= t99] = w_tail
    dtrain = xgb.DMatrix(Xg_tr[mtr], label=res_train[mtr], weight=w)
    if xgb_params is None:
        xgb_params = {"objective":"reg:pseudohubererror","max_depth":6,"eta":0.05,
                      "subsample":0.80,"colsample_bytree":0.80,"lambda":1.0,
                      "tree_method":"hist","seed":seed,"verbosity":0}
    booster = xgb.train(xgb_params, dtrain, num_boost_round=int(num_boost_round))
    res_te_all = booster.predict(xgb.DMatrix(Xg_te))
    final = base["pred_test_all"].copy()
    final[mte] = final[mte] + res_te_all[mte]
    return {"y_true":base["y_test_all"][mte],
            "y_pred_graph":base["pred_test_all"][mte],
            "y_pred_final":final[mte],
            "gmodel":base["model"], "hmodel":booster}

def choose_best_tau_dynamic(tr, va, ctx, tau_list):
    best_tau, best_mae = None, np.inf
    rows = []
    for tau in tau_list:
        if tau >= tr["X"].shape[1]: continue
        res = fit_huber_graph(tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau)
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau":tau,"val_MAE":mae})
        if mae < best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(tr, va, A_static, tau_list):
    best_tau, best_mae = None, np.inf
    rows = []
    for tau in tau_list:
        if tau >= tr["X"].shape[1]: continue
        m = Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=500))])
        res = fit_static_graph_regressor(m,tr["X"],tr["Y"],tr["M"],
                                          va["X"],va["Y"],va["M"],A_static,tau)
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau":tau,"val_MAE":mae})
        if mae < best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params, tau, tr, va, ctx):
    (alpha,eps,log_ha,max_depth,eta,subsample,colsample,
     reg_lambda,nround,w_mid,w_danger,w_tail) = params
    hybrid = fit_huber_hybrid(
        tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau,
        alpha=float(alpha), eps=float(eps), huber_alpha=10**float(log_ha),
        xgb_params={"objective":"reg:pseudohubererror",
                    "max_depth":int(round(max_depth)),"eta":float(eta),
                    "subsample":float(subsample),"colsample_bytree":float(colsample),
                    "lambda":float(reg_lambda),"tree_method":"hist",
                    "seed":RANDOM_SEED,"verbosity":0},
        num_boost_round=int(round(nround)), seed=RANDOM_SEED,
        w_mid=float(w_mid), w_danger=float(w_danger), w_tail=float(w_tail))
    mae = mean_absolute_error(hybrid["y_true"], hybrid["y_pred_final"])
    thr = np.percentile(hybrid["y_true"], 99)
    idx = hybrid["y_true"] >= thr
    if idx.sum() > 0:
        mfb,_ = compute_mfb_nmse(hybrid["y_true"][idx], hybrid["y_pred_final"][idx])
        return mae + 10.0*abs(mfb)
    return mae

def jso_optimize(tr, va, ctx, tau, bounds, n_pop=12, iters=15, seed=1):
    np.random.seed(seed)
    dim = bounds.shape[0]
    pop = bounds[:,0] + np.random.rand(n_pop,dim)*(bounds[:,1]-bounds[:,0])
    fit = np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in pop])
    best_p = pop[np.argmin(fit)].copy()
    best_f = float(fit.min())
    print(f"  initial best={best_f:.4f}")
    for it in range(iters):
        c = 1.0 - it/max(iters,1)
        new_pop = pop.copy()
        for i in range(n_pop):
            if np.random.rand() < 0.5:
                cand = pop[i] + np.random.randn(dim)*c*0.1 + c*(best_p-pop[i])*np.random.rand(dim)
            else:
                j = np.random.randint(0,n_pop)
                cand = pop[i] + (pop[j]-pop[i])*(np.random.rand(dim)-0.5)*c
            new_pop[i] = np.clip(cand, bounds[:,0], bounds[:,1])
        new_fit = np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in new_pop])
        better = new_fit < fit
        pop[better] = new_pop[better]; fit[better] = new_fit[better]
        if fit.min() < best_f: best_f=float(fit.min()); best_p=pop[np.argmin(fit)].copy()
        print(f"  iter {it+1:02d}/{iters} best={best_f:.4f}")
    return best_p, best_f

def add_result_row(rows, tail_rows, H, split, model, y_true, y_pred, extra=None):
    s = summarize_regression(y_true, y_pred)
    row = {"H":H,"split":split,"model":model,
           "MAE":s["MAE"],"RMSE":s["RMSE"],"R2":s["R2"],"n_test":len(y_true)}
    if extra: row.update(extra)
    rows.append(row)
    for p,thr,mae,rmse,n_tail in compute_tail_metrics(y_true, y_pred):
        tr = {"H":H,"split":split,"model":model,"percentile":p,
              "threshold":float(thr),"tail_MAE":float(mae) if pd.notna(mae) else np.nan,
              "tail_RMSE":float(rmse) if pd.notna(rmse) else np.nan,"n_tail":int(n_tail)}
        if extra: tr.update(extra)
        tail_rows.append(tr)

# ============================================================
# CELL 5 — BUILD LEAKAGE-FREE RAW TENSOR  (Beijing-specific)
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)

df = load_beijing_aqi()

station_ids = sorted(df["station_id"].unique())
N = len(station_ids)

FEATURES   = ["pm25", "temp", "pres", "dewp", "u_wind", "v_wind"]
TARGET_COL = "pm25"
u_idx = FEATURES.index("u_wind")
v_idx = FEATURES.index("v_wind")
target_idx = FEATURES.index(TARGET_COL)

all_times = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")

base = pd.MultiIndex.from_product(
    [all_times, station_ids], names=["timestamp","station_id"]
).to_frame(index=False)

aligned = base.merge(
    df[["timestamp","station_id"] + FEATURES],
    on=["timestamp","station_id"], how="left"
)

X_feat = []
for feat in FEATURES:
    mat = aligned.pivot(index="timestamp", columns="station_id",
                        values=feat).reindex(all_times)[station_ids]
    X_feat.append(mat.to_numpy(dtype=np.float32))
X_all_raw = np.stack(X_feat, axis=-1)                  # [T, N, F]

target_raw = aligned.pivot(
    index="timestamp", columns="station_id",
    values=TARGET_COL).reindex(all_times)[station_ids].to_numpy(dtype=np.float32)
Y_mask_full = (~np.isnan(target_raw)).astype(np.float32)

train_mask = all_times < SPLIT_TIME
fill_values = build_fill_values_from_train_timeline(X_all_raw[train_mask])

print(f"Tensor shape: {X_all_raw.shape}  "
      f"Target non-null: {Y_mask_full.sum():.0f}/{Y_mask_full.size:.0f}")

# ============================================================
# CELL 6 — BUILD GRAPH (identical logic, Beijing coords)
# ============================================================
nodes = build_nodes_from_coords(station_ids, STATION_COORDS)
edges_df = build_edges_from_nodes(nodes, k=GRAPH_K)
A_static = build_static_adj(nodes, k=GRAPH_K)

src = edges_df["src"].to_numpy(dtype=int)
dst = edges_df["dst"].to_numpy(dtype=int)
w_dist = edges_df["w_dist"].to_numpy(dtype=np.float32)
src_lat = nodes.loc[src,"lat"].to_numpy()
src_lon = nodes.loc[src,"lon"].to_numpy()
dst_lat = nodes.loc[dst,"lat"].to_numpy()
dst_lon = nodes.loc[dst,"lon"].to_numpy()
edge_bearing = bearing_radians(src_lat, src_lon, dst_lat, dst_lon)

graph_ctx = {
    "src": src, "dst": dst, "w_dist": w_dist,
    "edge_bearing": edge_bearing,
    "u_idx": u_idx, "v_idx": v_idx, "N": N
}

# ============================================================
# CELL 7 — SLIDING WINDOW CONSTRUCTION
# ============================================================
def build_windows(X_raw, Y_raw, Y_mask, all_times, split_time, L, H_list):
    """Build sliding-window tensors for train and test split."""
    n_times = len(all_times)
    data = {}
    for H in H_list:
        X_wins, Y_wins, M_wins, ttimes = [], [], [], []
        for t in range(L, n_times - H):
            t_target = t + H
            if all_times[t_target] < split_time:
                # train window
                pass  # collected below
            x_win = X_raw[t-L:t]          # [L, N, F]
            y_win = Y_raw[t_target]        # [N]
            m_win = Y_mask[t_target]       # [N]
            X_wins.append(x_win)
            Y_wins.append(y_win)
            M_wins.append(m_win)
            ttimes.append(all_times[t_target])

        X_arr = np.stack(X_wins, axis=0)   # [S, L, N, F]
        Y_arr = np.stack(Y_wins, axis=0)   # [S, N]
        M_arr = np.stack(M_wins, axis=0)   # [S, N]
        t_arr = np.array(ttimes)

        test_mask = t_arr >= split_time
        train_mask = ~test_mask

        X_train_raw = X_arr[train_mask]
        X_test_raw  = X_arr[test_mask]

        # Impute with train-fill-values only
        X_train_imp = impute_windows(X_train_raw, fill_values)
        X_test_imp  = impute_windows(X_test_raw,  fill_values)

        data[H] = {
            "X_train_imp": X_train_imp,
            "X_test_imp":  X_test_imp,
            "Y_train":     Y_arr[train_mask],
            "Y_test":      Y_arr[test_mask],
            "M_train":     M_arr[train_mask],
            "M_test":      M_arr[test_mask],
            "target_times_train": t_arr[train_mask],
            "target_times_test":  t_arr[test_mask],
        }
        print(f"H={H:2d}h  train={train_mask.sum():5d}  test={test_mask.sum():5d}")

    return data

print("Building windows...")
all_data = build_windows(X_all_raw, target_raw, Y_mask_full,
                         all_times, SPLIT_TIME, L, H_LIST)

# ============================================================
# CELL 8 — JSO BOUNDS (same as Chennai)
# ============================================================
bounds = np.array([
    [1.0,   8.0],    # alpha (gate sharpness)
    [0.01,  0.20],   # eps (gate floor)
    [-5.0, -2.0],    # log10(huber_alpha)
    [3.0,   8.0],    # max_depth
    [0.02,  0.15],   # eta
    [0.60,  1.00],   # subsample
    [0.60,  1.00],   # colsample
    [0.10,  5.0],    # lambda
    [200.0, 600.0],  # nround
    [1.5,   4.0],    # w_mid
    [3.0,   8.0],    # w_danger
    [6.0,  20.0],    # w_tail
])

# ============================================================
# CELL 9 — MAIN TRAINING LOOP (identical to Chennai)
# ============================================================
results_rows, tail_rows, tau_rows = [], [], []
all_models = {}

for H in H_LIST:
    import time as _time
    _h_start = _time.time()
    print(f"\n{'='*55}\nHORIZON H={H}h  [started at {_time.strftime('%H:%M:%S')}]\n{'='*55}")
    pack = all_data[H]
    Xtr = pack["X_train_imp"]; Ytr = pack["Y_train"]; Mtr = pack["M_train"]
    Xte = pack["X_test_imp"];  Yte = pack["Y_test"];  Mte = pack["M_test"]

    tr, va = train_val_split_time_order(
        Xtr, Ytr, Mtr, pack["target_times_train"], frac=VALID_FRAC)

    best_tau_dyn,    tau_tbl_dyn    = choose_best_tau_dynamic(tr, va, graph_ctx, TAU_LIST)
    best_tau_static, tau_tbl_static = choose_best_tau_static(tr, va, A_static, TAU_LIST)
    tau_rows.extend(tau_tbl_dyn.assign(H=H, family="dynamic").to_dict("records"))
    tau_rows.extend(tau_tbl_static.assign(H=H, family="static").to_dict("records"))
    print(f"tau_dyn={best_tau_dyn}  tau_static={best_tau_static}")

    all_models[H] = {"tau_dyn": best_tau_dyn, "tau_static": best_tau_static}

    # No-graph baselines
    for name, model in [
        ("Ridge",  Pipeline([("sc",StandardScaler()),("r",Ridge(alpha=10.0))])),
        ("Huber",  Pipeline([("sc",StandardScaler()),
                              ("h",HuberRegressor(epsilon=1.35,max_iter=1000))])),
        ("RF",     RandomForestRegressor(n_estimators=100,max_depth=8,   # reduced from 200/12 — saves ~20 min/horizon
                                          n_jobs=-1,random_state=RANDOM_SEED)),
        ("XGB",    XGBRegressor(n_estimators=200,max_depth=8,learning_rate=0.07,
                                 subsample=0.9,colsample_bytree=0.8,
                                 tree_method="hist",n_jobs=-1,
                                 random_state=RANDOM_SEED)),
    ]:
        res = fit_flat_regressor(model, Xtr,Ytr,Mtr, Xte,Yte,Mte)
        add_result_row(results_rows, tail_rows, H, "test", name,
                       res["y_true"], res["y_pred"])
        print(f"{name} done")

    # Static-graph baselines
    for name, model in [
        ("SG-Ridge", Pipeline([("sc",StandardScaler()),("r",Ridge(alpha=10.0))])),
        ("SG-Huber", Pipeline([("sc",StandardScaler()),
                                ("h",HuberRegressor(epsilon=1.35,max_iter=1000))])),
    ]:
        res = fit_static_graph_regressor(
            model, Xtr,Ytr,Mtr, Xte,Yte,Mte,
            A_static=A_static, tau=best_tau_static)
        add_result_row(results_rows, tail_rows, H, "test", name,
                       res["y_true"], res["y_pred"], extra={"tau":best_tau_static})
        print(f"{name} done")

    # Dynamic graph models
    res = fit_huber_graph(Xtr,Ytr,Mtr, Xte,Yte,Mte,
                          graph_ctx, tau=best_tau_dyn)
    add_result_row(results_rows, tail_rows, H, "test", "Huber-Graph",
                   res["y_true"], res["y_pred"], extra={"tau":best_tau_dyn})
    all_models[H]["graph"] = res["model"]
    print("Huber-Graph done")

    res = fit_huber_hybrid(Xtr,Ytr,Mtr, Xte,Yte,Mte,
                           graph_ctx, tau=best_tau_dyn,
                           num_boost_round=400, seed=RANDOM_SEED)
    add_result_row(results_rows, tail_rows, H, "test", "Hybrid",
                   res["y_true"], res["y_pred_final"], extra={"tau":best_tau_dyn})
    print("Hybrid done")

    if RUN_TUNED:
        best_p, best_obj = jso_optimize(tr, va, graph_ctx, best_tau_dyn,
                                         bounds, JSO_POP, JSO_ITERS, RANDOM_SEED)
        (al,ep,lha,md,eta,sub,col,lam,nr,wm,wd_w,wt) = best_p
        ha = 10**float(lha)

        res_tg = fit_huber_graph(Xtr,Ytr,Mtr, Xte,Yte,Mte, graph_ctx,
                                 tau=best_tau_dyn, alpha=float(al), eps=float(ep),
                                 huber_alpha=ha)
        add_result_row(results_rows, tail_rows, H, "test", "Tuned-Huber-Graph",
                       res_tg["y_true"], res_tg["y_pred"],
                       extra={"tau":best_tau_dyn,"val_obj":float(best_obj)})

        res_th = fit_huber_hybrid(Xtr,Ytr,Mtr, Xte,Yte,Mte, graph_ctx,
                                  tau=best_tau_dyn, alpha=float(al), eps=float(ep),
                                  huber_alpha=ha,
                                  xgb_params={
                                      "objective":"reg:pseudohubererror",
                                      "max_depth":int(round(md)),"eta":float(eta),
                                      "subsample":float(sub),"colsample_bytree":float(col),
                                      "lambda":float(lam),"tree_method":"hist",
                                      "seed":RANDOM_SEED,"verbosity":0},
                                  num_boost_round=int(round(nr)), seed=RANDOM_SEED,
                                  w_mid=float(wm), w_danger=float(wd_w), w_tail=float(wt))
        add_result_row(results_rows, tail_rows, H, "test", "Tuned-Hybrid",
                       res_th["y_true"], res_th["y_pred_final"],
                       extra={"tau":best_tau_dyn,"val_obj":float(best_obj)})
        print("Tuned models done")
    print(f"  H={H}h total wall time: {{(_time.time()-_h_start)/60:.1f}} min")

    # ── Checkpoint: save after each horizon (timeout protection) ──
    import pandas as _cpd
    _cpd.DataFrame(results_rows).sort_values(["H","MAE"]).reset_index(drop=True).to_csv(
        f"{OUT_DIR}/ckpt_results_H{H}.csv", index=False)
    _cpd.DataFrame(tail_rows).to_csv(f"{OUT_DIR}/ckpt_tail_H{H}.csv", index=False)
    print(f"  ✓ Checkpoint saved -> {OUT_DIR}/ckpt_results_H{H}.csv")

# ============================================================
# CELL 10 — EXCEEDANCE HEAD (identical)
# ============================================================
exceed_rows = []
for H in H_LIST:
    pack = all_data[H]
    tau  = all_models[H]["tau_dyn"]
    Ztr = make_graph_features_dynamic(pack["X_train_imp"], graph_ctx, tau=tau)
    Zte = make_graph_features_dynamic(pack["X_test_imp"],  graph_ctx, tau=tau)
    Xc_tr = Ztr.reshape(-1, Ztr.shape[-1])
    Xc_te = Zte.reshape(-1, Zte.shape[-1])
    ytr = pack["Y_train"].reshape(-1); yte = pack["Y_test"].reshape(-1)
    mtr = pack["M_train"].reshape(-1)>0.5; mte = pack["M_test"].reshape(-1)>0.5
    thr = np.percentile(ytr[mtr], 95)
    clf = Pipeline([("sc",StandardScaler()),
                    ("logit",LogisticRegression(max_iter=1000,class_weight="balanced"))])
    clf.fit(Xc_tr[mtr], (ytr[mtr]>=thr).astype(int))
    prob = clf.predict_proba(Xc_te[mte])[:,1]
    pred = (prob>=0.5).astype(int)
    yte_cls = (yte[mte]>=thr).astype(int)
    pr,rc,f1,_ = precision_recall_fscore_support(yte_cls, pred, average="binary", zero_division=0)
    ap = average_precision_score(yte_cls, prob)
    exceed_rows.append({"H":H,"threshold_95pct":float(thr),
                         "precision":float(pr),"recall":float(rc),
                         "f1":float(f1),"pr_auc":float(ap),
                         "n_pos":int(yte_cls.sum()),"n_total":int(len(yte_cls))})

# ============================================================
# CELL 11 — SAVE RESULTS
# ============================================================
results_df = pd.DataFrame(results_rows).sort_values(["H","MAE"]).reset_index(drop=True)
tail_df    = pd.DataFrame(tail_rows).sort_values(["H","model","percentile"]).reset_index(drop=True)
tau_df     = pd.DataFrame(tau_rows).sort_values(["H","family","val_MAE"]).reset_index(drop=True)
exceed_df  = pd.DataFrame(exceed_rows).sort_values("H").reset_index(drop=True)

results_df.to_csv(f"{OUT_DIR}/results_main.csv",    index=False)
tail_df.to_csv(   f"{OUT_DIR}/results_tail.csv",    index=False)
tau_df.to_csv(    f"{OUT_DIR}/tau_search.csv",      index=False)
exceed_df.to_csv( f"{OUT_DIR}/exceedance.csv",      index=False)
joblib.dump(all_models, f"{OUT_DIR}/models.pkl")

print("\n=== MAIN RESULTS ==="); display(results_df)
print("\n=== TAIL (99th pct) ===")
display(tail_df[tail_df.percentile==99][["H","model","tail_MAE"]].dropna())
print("\n=== EXCEEDANCE HEAD ==="); display(exceed_df)
print(f"\nAll outputs saved to {OUT_DIR}/")

  Reading CSVs from: /kaggle/input/datasets/sid321axn/beijing-multisite-airquality-data-set
Loaded 420,768 rows, 12 stations, period 2013-03-01 00:00:00 → 2017-02-28 23:00:00
Tensor shape: (35064, 12, 6)  Target non-null: 412029/420768
Building windows...
H= 1h  train=30695  test= 4344
H= 3h  train=30693  test= 4344
H= 6h  train=30690  test= 4344
H=12h  train=30684  test= 4344
H=24h  train=30672  test= 4344

HORIZON H=1h  [started at 09:31:43]
tau_dyn=0  tau_static=0
Ridge done
Huber done
RF done
XGB done
SG-Ridge done
SG-Huber done
Huber-Graph done
Hybrid done
  initial best=15.4080
  iter 01/8 best=15.4080
  iter 02/8 best=15.4080
  iter 03/8 best=15.4080
  iter 04/8 best=15.4080
  iter 05/8 best=15.3961
  iter 06/8 best=15.3961
  iter 07/8 best=15.3961
  iter 08/8 best=15.3812
Tuned models done
  H=1h total wall time: {(_time.time()-_h_start)/60:.1f} min
  ✓ Checkpoint saved -> dghxh_beijing_aqi/ckpt_results_H1.csv

HORIZON H=3h  [started at 10:18:05]
tau_dyn=0  tau_static=0
Ridge d

,H,split,model,MAE,RMSE,R2,n_test,tau,val_obj
0,1,test,Hybrid,17.442974,33.087132,0.887955,51063,0.0,NaN
1,1,test,Tuned-Hybrid,17.572129,33.443537,0.885528,51063,0.0,15.381167
2,1,test,XGB,17.848146,33.693390,0.883811,51063,NaN,NaN
3,1,test,Huber,18.209650,34.577549,0.877633,51063,NaN,NaN
4,1,test,RF,18.295458,34.204558,0.880259,51063,NaN,NaN
5,1,test,Tuned-Huber-Graph,18.428872,34.440395,0.878602,51063,0.0,15.381167
6,1,test,SG-Huber,18.434420,34.425118,0.878710,51063,0.0,NaN
7,1,test,Huber-Graph,18.447626,34.509396,0.878115,51063,0.0,NaN
8,1,test,Ridge,18.990686,34.294109,0.879631,51063,NaN,NaN
9,1,test,SG-Ridge,19.055803,34.254984,0.879906,51063,0.0,NaN



=== TAIL (99th pct) ===


,H,model,tail_MAE
2,1,Huber,73.369232
5,1,Huber-Graph,74.922915
8,1,Hybrid,68.024354
11,1,RF,87.591916
14,1,Ridge,84.431412
17,1,SG-Huber,75.242117
20,1,SG-Ridge,86.498291
23,1,Tuned-Huber-Graph,74.980641
26,1,Tuned-Hybrid,66.337615
29,1,XGB,102.783287



=== EXCEEDANCE HEAD ===


,H,threshold_95pct,precision,recall,f1,pr_auc,n_pos,n_total
0,1,235.0,0.486694,0.971197,0.648438,0.871844,4444,51063
1,3,235.0,0.395291,0.936994,0.556015,0.759238,4444,51063
2,6,235.0,0.330924,0.896490,0.483407,0.621572,4444,51063
3,12,235.0,0.267650,0.873537,0.409753,0.473484,4444,51063
4,24,235.0,0.209184,0.869262,0.337218,0.357150,4444,51063



All outputs saved to dghxh_beijing_aqi/
